### ЗАДАЧА: Пакетная загрузка накладных (try/except + custom exceptions)

Из внешней системы приходят строки с накладными.
Нужно безопасно разобрать их, отделить валидные записи от ошибок и собрать краткий отчёт.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `InvoiceError`
   - `RowFormatError`
   - `QuantityError`
   - `PriceError`
   - `StatusError`.

2. Функцию `parse_invoice(row)`:
   - формат: `invoice_id,item,quantity,price,status`
   - `quantity` должен быть целым числом и `> 0`
   - `price` должен быть числом и `> 0`
   - допустимые статусы: `new`, `approved`, `paid`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `load_invoices(rows)`:
   - вернуть `(invoices, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных накладных,
   - ошибки по типам,
   - сумму только для накладных со статусом `paid`,
   - товар-лидер по суммарному количеству в валидных накладных.

ПОДСКАЗКИ:
- `quantity` и `price` сначала конвертируются, потом валидируются.
- Для ошибок по типам удобно собирать обычный словарь-счётчик.


In [ ]:
rows = [
    'INV-100,Keyboard,3,120,paid',
    'INV-101,Mouse,0,40,new',
    'INV-102,Monitor,2,abc,approved',
    'INV-103,Laptop,1,1400,shipped',
    'INV-104,Keyboard,5,110,paid',
    'INV-105,Dock,2,-50,approved',
]

class InvoiceError(Exception):
    pass

class RowFormatError(InvoiceError):
    pass

class QuantityError(InvoiceError):
    pass

class PriceError(InvoiceError):
    pass

class StatusError(InvoiceError):
    pass

def parse_invoice(row):
    parts = row.split(",")
    if len(parts) != 5:
        raise RowFormatError('в строке не 5 элементов')
    invoice_id, item, quantity, price, status = parts

    try:
        quantity = int(quantity)
    except ValueError as e:
        raise QuantityError('не является числом') from e
    if quantity < 0:
        raise QuantityError('количество не может быть отрицательным')

    try:
        price = float(price) 
    except ValueError as e:
        raise PriceError('некорректная цена товара') from e
    if price < 0:
        raise PriceError('цена не может быть отрицательной') 

    valid_statuses = {'paid', 'new', 'approved'} 
    if status not in valid_statuses:
        raise StatusError(f"Недопустимый статус: '{status}'. Допустимые: {valid_statuses}")

    return {
        'invoice_id': invoice_id,
        'product': item,
        'quantity': quantity,
        'price': price,
        'status': status
    }

def load_invoices(rows):
    invoices = []
    errors = []

    for i, row in enumerate(rows):  
        try:
            invoices.append(parse_invoice(row))
        except InvoiceError as e:
            errors.append({
                'row_index': i,
                'row_content': row,
                'error_type': type(e).__name__,
                'error_message': str(e)
            })
    return invoices, errors

invoices, errors = load_invoices(rows)

print(f"Валидных накладных: {len(invoices)}")
print(f"Ошибок: {len(errors)}")


if errors:
    print("\nОшибки по типам:")
    error_counts = {}
    for error in errors:
        error_type = error['error_type']
        error_counts[error_type] = error_counts.get(error_type, 0) + 1

    for error_type, count in error_counts.items():
        print(f"  {error_type}: {count} ошибок")

    print("\nДетали ошибок:")
    for error in errors:
        print(f"  Строка {error['row_index']}: {error['error_message']} (содержимое: '{error['row_content']}')")
else:
    print("Ошибок нет.")

paid_total = sum(invoice['quantity'] * invoice['price'] for invoice in invoices if invoice['status'] == 'paid')
print(f"\nОбщая сумма оплаченных накладных (paid_total): {paid_total}")

product_quantity = {}
for invoice in invoices:
    product = invoice['product']
    quantity = invoice['quantity']
    product_quantity[product] = product_quantity.get(product, 0) + quantity

if product_quantity:
    leader_product = max(product_quantity, key=product_quantity.get)
    leader_quantity = product_quantity[leader_product]
    print(f"Товар-лидер по количеству: '{leader_product}' ({leader_quantity} шт.)")
else:
    print("Нет валидных накладных для анализа товаров.")
